# 01.4 — Byte Pair Encoding (BPE)

**Problem it solves:** Word-level vocab has OOV. Character-level vocab has very long sequences. BPE finds the optimal middle ground: common words get single tokens, rare words get split into subword pieces you've seen before.

**Why it was invented:** Originally a data compression algorithm (Philip Gage, 1994). Sennrich et al. (2016) adapted it for NMT. GPT-2/3/4, Llama, Mistral — all use BPE or a close variant (BBPE = byte-level BPE).

**Core idea:** Start with characters. Repeatedly merge the most frequent pair of adjacent tokens until you hit your vocabulary size limit.

---

## Part 1: BPE From Scratch

In [ ]:
from collections import Counter, defaultdict
import re

def get_vocab(corpus: list[str]) -> dict:
    """Convert corpus to word-frequency dict with characters separated by spaces.
    We append </w> to mark word endings — crucial so 'est</w>' != 'est' mid-word."""
    vocab = Counter()
    for text in corpus:
        for word in text.lower().split():
            # Represent each word as a tuple of characters + end-of-word marker
            vocab[' '.join(list(word) + ['</w>'])] += 1
    return dict(vocab)

def get_pairs(vocab: dict) -> Counter:
    """Count all adjacent symbol pairs across the vocab."""
    pairs = Counter()
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i+1])] += freq
    return pairs

def merge_pair(pair: tuple, vocab: dict) -> dict:
    """Merge all occurrences of pair in vocab."""
    new_vocab = {}
    bigram = re.escape(' '.join(pair))
    pattern = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
    replacement = ''.join(pair)
    for word in vocab:
        new_word = pattern.sub(replacement, word)
        new_vocab[new_word] = vocab[word]
    return new_vocab

def train_bpe(corpus: list[str], num_merges: int) -> tuple[dict, list]:
    """Train BPE. Returns final vocab and list of merge rules."""
    vocab = get_vocab(corpus)
    merges = []
    
    print(f"Initial vocab size: {len(set(sym for w in vocab for sym in w.split()))} symbols")
    print(f"Initial tokens (sample): {list(vocab.items())[:3]}")
    print()
    
    for i in range(num_merges):
        pairs = get_pairs(vocab)
        if not pairs:
            break
        best = max(pairs, key=pairs.get)
        vocab = merge_pair(best, vocab)
        merges.append(best)
        if i < 10 or i % 50 == 0:
            print(f"Merge {i+1:3d}: {best[0]!r} + {best[1]!r} -> {best[0]+best[1]!r}  (freq={pairs[best]})")
    
    return vocab, merges

corpus = [
    "low low low low low lower lower newest newest newest newest newest newest newest widest widest widest",
    "newer newer newer newer newer newer",
]

vocab, merges = train_bpe(corpus, num_merges=20)

In [ ]:
# Encode new text using learned merge rules
def encode_bpe(word: str, merges: list) -> list[str]:
    """Apply learned BPE merges to a new word."""
    # Start: each char is a separate token, with </w> at end
    tokens = list(word) + ['</w>']
    
    # Apply merges in the order they were learned
    for (a, b) in merges:
        i = 0
        new_tokens = []
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == a and tokens[i+1] == b:
                new_tokens.append(a + b)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    return tokens

test_words = ['low', 'lower', 'newest', 'widest', 'newer', 'newes', 'lowest']
print("Tokenization with learned BPE merges:")
for w in test_words:
    print(f"  {w:12} -> {encode_bpe(w, merges)}")

## Part 2: BPE on a Real Corpus

In [ ]:
# Use a slightly larger corpus to see BPE in action
import urllib.request, json

sample_text = """
the cat sat on the mat the cat ate the rat the rat ran fast
natural language processing is a field of artificial intelligence
machine learning models learn from data to make predictions
transformers use attention mechanisms to process sequences
tokenization splits text into smaller units called tokens
byte pair encoding merges frequent character pairs iteratively
vocabulary size is a hyperparameter that trades coverage for speed
subword tokenization handles out of vocabulary words gracefully
modern language models use byte level byte pair encoding
the quick brown fox jumps over the lazy dog
""".strip().split('\n')

print(f"Corpus: {len(sample_text)} sentences")
print(f"Unique words: {len(set(w for s in sample_text for w in s.split()))}")

vocab_50, merges_50 = train_bpe(sample_text, num_merges=50)

# Show final token inventory
final_symbols = set(sym for word in vocab_50 for sym in word.split())
print(f"\nFinal vocab symbols: {len(final_symbols)}")
print(f"Symbols (sorted by length): {sorted(final_symbols, key=len, reverse=True)[:20]}")

In [ ]:
# Compare tokenization at different merge counts
_, merges_10 = train_bpe(sample_text, num_merges=10)
_, merges_30 = train_bpe(sample_text, num_merges=30)

words_to_test = ['tokenization', 'transformers', 'intelligence', 'cat', 'unknown']
print("\nTokenization at different merge counts:")
print(f"{'Word':20} {'10 merges':30} {'30 merges':30} {'50 merges'}")
print('-' * 100)
for w in words_to_test:
    t10 = encode_bpe(w, merges_10)
    t30 = encode_bpe(w, merges_30)
    t50 = encode_bpe(w, merges_50)
    print(f"{w:20} {str(t10):30} {str(t30):30} {t50}")

## Part 3: Byte-Level BPE (BBPE)

GPT-2 uses BBPE — operates on raw bytes instead of characters. This means:
- Zero OOV — any byte sequence is encodeable  
- Works on any language without a language-specific character set
- Handles emoji, code, math, everything

In [ ]:
def get_byte_vocab(corpus: list[str]) -> dict:
    """Like get_vocab but operates on UTF-8 bytes."""
    vocab = Counter()
    for text in corpus:
        for word in text.lower().split():
            # Encode to bytes, represent each byte as a symbol
            byte_seq = word.encode('utf-8')
            symbols = [str(b) for b in byte_seq] + ['</w>']
            vocab[' '.join(symbols)] += 1
    return dict(vocab)

# Demonstrate on text with non-ASCII
multilingual = ["hello world café naïve", "こんにちは 世界", "مرحبا"]

byte_vocab = get_byte_vocab(multilingual)
print("Byte-level representation:")
for word_repr, freq in list(byte_vocab.items())[:6]:
    print(f"  freq={freq}  {word_repr}")

print("\nKey advantage: zero OOV possible — every string is encodeable")

## Part 4: What Breaks BPE?

In [ ]:
# 1. BPE is greedy — does not guarantee globally optimal splits
# Different training corpora produce different tokenizations of the same word

# 2. Token boundary sensitivity: models learn token-internal patterns
# 'un' + 'known' vs 'unk' + 'nown' can produce different model behavior

# 3. Number tokenization is awful
_, merges_num = train_bpe(["1 2 3 4 5 10 11 12 100 101 1000 1001 2024 2025"], 20)
for num in ['100', '101', '102', '999', '2024', '2025']:
    print(f"  {num:6} -> {encode_bpe(num, merges_num)}")

print("\n3. Numbers get weird splits because their frequency isn't compositional.")
print("This is one reason LLMs struggle at arithmetic.")

# 4. Whitespace prefix matters
# In GPT tokenization: ' cat' and 'cat' are DIFFERENT tokens
# The leading space is part of the token
print("\n4. Whitespace prefix: ' cat' != 'cat' in GPT tokenizers")
print("This means the model's behavior can change based on where a word appears.")

## Summary

**What it does:** Builds a subword vocabulary by iteratively merging the most frequent adjacent symbol pairs.

**Why it's the default:** Balances vocabulary size, sequence length, and OOV coverage. A 50k BPE vocab covers virtually all English text.

**What breaks it:**
- Numbers and codes tokenize poorly (frequency ≠ semantic structure)
- Language-specific morphology may be ignored (works better for English than Turkish)
- Different tokenizers for different models mean embeddings are not portable
- BPE is deterministic — SentencePiece offers stochastic segmentation for regularization

---
**Next:** 01.5 — Stemming & Lemmatization